In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
os.chdir('/Users/jinghuang/Desktop/SSTP/Parse')

In [ ]:
def flatten_labels(label_tree, nones=True):
    flat = []
    for category, tests in label_tree.items():
        if nones: 
            flat.append((category,'', ''))
        else:
            flat.append((category,))
        if isinstance(tests, list):
            for t in tests:
                flat.append((category, t))
        else:                       # dict
            for test_name, subs in tests.items():
                if subs:
                    for sub in subs:
                        flat.append((category, test_name, sub))
                else:
                    if nones:
                        flat.append((category, test_name, ''))
                    else:
                        flat.append((category, test_name))
    return flat

def label_strings(flat_labels: list[tuple[str, str, str | None]]) -> list[str]:
    """Render each (cat, test, sub) triple as a single string for matching."""
    return [" | ".join(part for part in triple if part) for triple in flat_labels]

In [ ]:
# Get patient
patient = Path('case_outputs/tables_clean/00001.csv')
patient = pd.read_csv(patient)

# Get field_labels
fl_path = Path('instructions/field_labels.json')
fl_flat = flatten_labels(json.loads(fl_path.read_text()), nones=False)

In [ ]:
idx1, idx2 = 0,0
res = []
while idx1 < patient.shape[0]-1:
    if idx2 < len(fl_flat):
        B = fl_flat[idx2]
    else:
        res.append((idx2, None))
        print(f"Not FOUND {idx1:02d} {idx2:02d}\t ")
        idx1 += 1
        idx2 = 0

    if len(B) == 1: # if category, needs to also match casewise 
        A = patient.iloc[idx1]['Left']
        B0 = B[0].upper()
    else:
        A = A.lower()
        B0 = B[-1].lower()
    if B0 in A:
        res.append((idx2, idx1))
        print(f"WE FOUND {idx1:02d} {idx2:02d} {B}\t {patient.iloc[idx1]['Left']}")
        idx1 += 1
        idx2 = 0
    else:
        idx2 += 1
    
    print(f"{idx1:02d} {idx2:02d}")

In [ ]:

def flatten_labels2(label_tree):
    flat = []
    for category, tests in label_tree.items():
        flat.append((category,'', ''))
        if isinstance(tests, list):
            for t in tests:
                flat.append((category, t, ''))
        else:                       # dict
            for test_name, subs in tests.items():
                if subs:
                    for sub in subs:
                        flat.append((category, test_name, sub))
                else:
                    flat.append((category, test_name, ''))
    return flat

fl_flat = flatten_labels2(json.loads(fl_path.read_text()))
multi_cols = pd.MultiIndex.from_tuples(
    fl_flat, names=["Category", "Test", "Subtest"]
)
data = np.random.randint(0, 100, size=(5, len(multi_cols)))
df = pd.DataFrame(data, columns=multi_cols)

file_name = 'report_with_multi_header.xlsx'

# Simply remove 'index=False'. This resolves the error.
df.to_excel(file_name)

print(f"✅ Excel file '{file_name}' was created successfully (with index column).")

In [ ]:
fl_flat